# Suivi d'expériences MLflow — TravelMatch

Ce notebook ne contient **aucune logique** : il appelle les fonctions du module `mlops/` (source de vérité, versionnée et testée). Il sert uniquement à lancer les entraînements et visualiser les résultats.

Pré-requis : serveur MLflow sur http://localhost:5000 (`docker compose up mlflow`). Sinon, bascule automatique sur `sqlite:///mlflow.db`.

Les deux expériences tracées :
- **TravelMatch_Clustering** : sélection de k (elbow, silhouette, Davies-Bouldin, Calinski-Harabasz)
- **TravelMatch_Classifier** : RandomizedSearchCV (learning curve, classification report, confusion matrix, importances)

In [ ]:
# On remonte à la racine du projet pour que les imports et les chemins de données fonctionnent
import os, sys
ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.append(ROOT)
os.chdir(ROOT)

from mlops.train import run_clustering, run_classification
from mlops.features import build_destination_features

## 1. Données utilisées
ADN d'activités enrichi : 560 villes × 9 dimensions (`DATA/raw/Worldwide_Travel_Cities.csv`). Voir `clustering_study.ipynb` pour l'étude détaillée.

In [ ]:
cities, X, descriptive = build_destination_features()
print(f"{len(cities)} villes, {X.shape[1]} dimensions d'activités")
X.head()

## 2. Expérience clustering — sélection de k
Logge un run imbriqué par k + les courbes de sélection et le diagramme de silhouette.

In [ ]:
best_k = run_clustering(k_min=2, k_max=12, final_k=6)
print("k retenu (métier) :", best_k)

## 3. Expérience classification — RandomizedSearchCV
On garde `best_k=6` pour rester aligné avec les 6 archétypes du produit (à arbitrer vs la silhouette).

In [ ]:
run_classification(best_k=6, n_iter=30)

## 4. Visualiser les résultats
Ouvrir l'UI MLflow : http://localhost:5000 → comparer les runs, les métriques et télécharger les artefacts (courbes, matrice de confusion, modèles).

Pour afficher une courbe directement ici, on peut aussi la régénérer via les helpers de `mlops.mlflow_utils`.